# Diff-MoE on BabyLM -- Kaggle T4 x2 training notebook

Runs one config from `docs/plan.md`'s 2x2 ablation (standard/differential attention x dense/MoE FFN) on Kaggle's dual-T4 notebook, using both GPUs via DDP. Training data is the [BabyLM Challenge](https://babylm.github.io/) corpus: six domains (child-directed speech, adult conversation, literary prose, subtitles, Wikipedia, telephone dialogue), released as one text file per domain rather than a single blob -- which is what makes it a meaningful testbed for whether MoE experts specialize.

**Before running**: set Notebook Settings -> Accelerator = **GPU T4 x2**.

**Session limits**: 12h max, ~30 GPU-h/week (same quota whether you pick T4 x1 or x2). Checkpoints save every `ckpt_freq` steps to `/kaggle/working/checkpoints` -- copy that folder to a Kaggle Dataset before your session ends so the next session can resume.

**Workflow per run**: (1) clone repo, (2) prepare tokenized data once and re-use across sessions via a Kaggle Dataset, (3) throughput probe on both GPUs, (4) train with DDP, (5) inspect metrics + report.

## 1. Setup

In [ ]:
!git clone -b rebuild https://github.com/ramprasathk07/Differential-MOE.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements.txt

In [ ]:
# Staleness guard: Kaggle keeps its OWN copy of this notebook, independent of the repo.
# If you edited/imported an older version, its cells can pass flags the freshly-cloned
# code no longer accepts (e.g. the old TinyStories-era --max_stories), which fails data
# prep and then dies later with a confusing FileNotFoundError on train.bin.
import subprocess

_help = subprocess.run(['python', '-m', 'src.data.train_tokenizer', '--help'],
                       capture_output=True, text=True).stdout
if '--track' in _help:
    print('OK: cloned code and this notebook agree (BabyLM --track interface present).')
else:
    raise SystemExit(
        'STALE NOTEBOOK/CODE MISMATCH: the cloned code does not expose --track.\n'
        'Re-import the current notebook: in the Kaggle editor use File > Import Notebook\n'
        'and upload notebooks/kaggle_train.ipynb from the repo (or copy its cells over).'
    )

In [ ]:
import torch
n_gpu = torch.cuda.device_count()
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', n_gpu)
for i in range(n_gpu):
    print(f'  cuda:{i}', torch.cuda.get_device_name(i))
if n_gpu < 2:
    print('\nWARNING: fewer than 2 GPUs visible -- set Notebook Settings > Accelerator = GPU T4 x2, '
          'then Session > Restart & Run All. The --ddp cells below need nproc_per_node=2 to match n_gpu.')

## 2. Run settings

Change these per run -- no code edits needed elsewhere in the notebook.

In [ ]:
import os

CONFIG = 'configs/a_dense.yaml'  # a_dense / a_diff / a_moe / a_diffmoe / b_final
WANDB_PROJECT = 'diff-moe-kaggle'  # change freely per experiment batch, no code changes needed
USE_WANDB = True
N_GPU = 2  # matches Accelerator = GPU T4 x2; set to 1 if you picked the single-T4 option

TRACK = 'strict' if 'b_final' in CONFIG else 'strict-small'  # BabyLM track: 10M words (tier A) or 100M words (tier B)
DATA_DIR = '/kaggle/working/data_b' if TRACK == 'strict' else '/kaggle/working/data'  # switch to '/kaggle/input/<dataset-name>' once tokenized once and re-uploaded

# wandb auth: without this, rank0's wandb.init() blocks at a login prompt inside the
# non-interactive training cell. Store your key once via Add-ons > Secrets > WANDB_API_KEY.
if USE_WANDB:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
        print('wandb key loaded from Kaggle secret WANDB_API_KEY')
    except Exception as e:
        print('WARNING: no WANDB_API_KEY Kaggle secret -- the training cell will hang at wandb login.')
        print('Fix: Add-ons > Secrets > add WANDB_API_KEY, or set USE_WANDB = False. Details:', e)

run_name = CONFIG.split('/')[-1].replace('.yaml', '')
wandb_flag = f'--wandb --wandb_project {WANDB_PROJECT}' if USE_WANDB else ''
ddp_prefix = f'torchrun --standalone --nproc_per_node={N_GPU}' if N_GPU > 1 else 'python'
ddp_flag = '--ddp' if N_GPU > 1 else ''
print('run_name:', run_name)
print('track:', TRACK, '| data_dir:', DATA_DIR)
print('launch prefix:', ddp_prefix)

## 3. Tokenizer + data

Run once, then attach the output as a Kaggle Dataset ("New Dataset" from notebook output) so later sessions can skip straight to training via `DATA_DIR` pointing at `/kaggle/input/<dataset-name>`. Tokenizing is CPU-only -- no benefit from N_GPU here. Downloads the six BabyLM domain files (bnc_spoken, childes, gutenberg, open_subtitles, simple_wiki, switchboard) for whichever `TRACK` the settings cell selected.

In [ ]:
import os
if not os.path.exists(f'{DATA_DIR}/train.bin'):
    # adaptive vocab sweep (docs/plan.md SS2) -- run once, read the fertility table, pick a vocab_size
    # BabyLM's six domains are more heterogeneous than a single-genre corpus, so this sweep
    # should be re-run rather than assuming a vocab size chosen for a different dataset still applies
    !python -m src.data.train_tokenizer --sweep --candidates 2048 4096 8192 16384 --track {TRACK} --max_lines_per_domain 20000
else:
    print('data already prepared at', DATA_DIR)

In [ ]:
VOCAB_SIZE = 4096  # set from the sweep table above -- must match the vocab_size in CONFIG's yaml

if not os.path.exists(f'{DATA_DIR}/train.bin'):
    !python -m src.data.train_tokenizer --vocab_size {VOCAB_SIZE} --out {DATA_DIR}/tokenizer.json --track {TRACK}
    !python -m src.data.prepare --tokenizer {DATA_DIR}/tokenizer.json --out_dir {DATA_DIR} --track {TRACK}

## 4. Throughput probe (docs/plan.md Phase 3)

Runs ~100 steps across both GPUs and reports tok/s so the token budget / wall-clock estimate is measured, not guessed. `batch_size` in the config is **per-GPU** -- with `N_GPU=2` the effective global batch doubles automatically (see `report.json`'s `effective_global_batch_tokens`).

In [ ]:
# NOTE: probe writes to its own out_dir so its checkpoints / LR history / 'best'
# selections don't pollute the real run (which would otherwise auto-resume from
# the probe's step-100 checkpoint with a mismatched cosine schedule).
!{ddp_prefix} -m src.train --config {CONFIG} --data_dir {DATA_DIR} --out_dir /kaggle/working/probe {ddp_flag} --max_steps 100

## 5. Full training run

Re-running this cell auto-resumes from `checkpoints/<run_name>/last.pt` if it exists -- safe to re-run after a Kaggle session restart. Rank-0-only logging/checkpointing/wandb is handled inside `src/train.py`; nothing extra needed here.

In [ ]:
!{ddp_prefix} -m src.train --config {CONFIG} --data_dir {DATA_DIR} --out_dir /kaggle/working/checkpoints {ddp_flag} {wandb_flag}

## 6. Inspect metrics + report

In [ ]:
import json
with open(f'/kaggle/working/checkpoints/{run_name}/report.json') as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f'/kaggle/working/checkpoints/{run_name}/metrics.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.dropna(subset=['train_loss']).plot(x='step', y='train_loss', ax=axes[0], title='train loss')
df.dropna(subset=['val_nll']).plot(x='step', y='val_nll', ax=axes[1], title='val NLL', marker='o')
plt.tight_layout()
plt.show()
df.tail(10)

## 7. Params table (for README / blog tables -- never hand-compute)

In [ ]:
!python -m src.params --config configs/a_dense.yaml configs/a_diff.yaml configs/a_moe.yaml configs/a_diffmoe.yaml

## 8. Persist checkpoints across sessions

Kaggle wipes `/kaggle/working` between sessions. 'Save Version' preserves everything under `/kaggle/working` as this notebook's output. To resume in a NEXT session: attach that output (or a Dataset made from it) as an input, then **copy it back into `/kaggle/working`** before training -- `/kaggle/input` is read-only, so pointing `--out_dir` at it directly would crash on the first checkpoint save.

In [ ]:
# This session's checkpoints (saved automatically when you 'Save Version'):
!ls -la /kaggle/working/checkpoints/{run_name}/ 2>/dev/null || echo 'no checkpoints yet'
print("best/ holds only the top-2 checkpoints (max_best_checkpoints in config); last.pt is always kept for resume.")

# NEXT session, to resume: attach the previous version's output as an input dataset,
# then copy it into the writable working dir BEFORE running the training cell
# (/kaggle/input is READ-ONLY -- training must never write there). Uncomment + edit:
# !mkdir -p /kaggle/working/checkpoints
# !cp -r /kaggle/input/<your-ckpt-dataset>/checkpoints/* /kaggle/working/checkpoints/